In [1]:
import pandas as pd
from tqdm import tqdm
import re

import numpy as np
import matplotlib.pyplot as plt
import openai
import pickle
import os

from typing import List, Dict, Tuple
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from kneed import KneeLocator

from sklearn.metrics import silhouette_score


from collections import defaultdict

import time

from collections import defaultdict, Counter

import gc, torch


In [2]:
openai.api_key ='xxx'
openai_client = openai.OpenAI(api_key=openai.api_key)


In [3]:
def save_to_pickle(obj, filepath):
    with open(filepath, 'wb') as f:
        pickle.dump(obj, f)

def load_from_pickle(filepath):
    with open(filepath, 'rb') as f:
        return pickle.load(f)

In [4]:

# ---------------------------------------------------
# 1. Embed Terms
# ---------------------------------------------------
def embed_terms(
    terms: List[str],
    model_name: str = "all-mpnet-base-v2"
) -> np.ndarray:
    """
    Embed a list of terms using SentenceTransformer.
    """
    model = SentenceTransformer(model_name, device="cuda:1")
    embeddings = model.encode(terms, convert_to_numpy=True)
    
    model = model.cpu()      # move weights off GPU
    del model                # remove reference
    # del embeddings         # only if you no longer need them
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    return embeddings


# ---------------------------------------------------
# 2. Evaluate Different K Values
# ---------------------------------------------------
def evaluate_k_range(embeddings, k_list, random_state=42):
    scores = []

    for k in tqdm(k_list):
        kmeans = KMeans(n_clusters=k, random_state=random_state, n_init="auto", init="k-means++")
        labels = kmeans.fit_predict(embeddings)

        score = silhouette_score(embeddings, labels, metric="cosine")
        scores.append(score)

    return k_list, scores
# ---------------------------------------------------
# 3. Find Optimal K Using KneeLocator
# ---------------------------------------------------
def find_optimal_k(
    k_values: List[int],
    scores: List[float]
) -> int:
    """
    Use kneed to detect elbow point.
    """
    kn = KneeLocator(
        k_values,
        scores,
        curve="concave",
        direction="increasing",
        S = 3
    )

    # Fallback if knee not detected
    if kn.knee is None:
        print("No knee detected")
        return k_values[np.argmax(scores)]

    return kn.knee


# ---------------------------------------------------
# 4. Final Clustering
# ---------------------------------------------------
def generate_final_clusters(
    terms: List[str],
    embeddings: np.ndarray,
    optimal_k: int,
    random_state: int = 42
) -> Dict[int, List[str]]:
    """
    Run KMeans with optimal_k and return clustered terms.
    """
    kmeans = KMeans(n_clusters=optimal_k, random_state=random_state, init="k-means++", n_init="auto")
    labels = kmeans.fit_predict(embeddings)

    clusters = {}
    for label, term in zip(labels, terms):
        clusters.setdefault(label, []).append(term)

    return clusters


# ---------------------------------------------------
# 5. Full Pipeline
# ---------------------------------------------------
def cluster_terms_pipeline(
    terms: List[str],
    k_list: List[int],
    figure_name: str,
    model_name: str = "all-mpnet-base-v2",
) -> Dict[int, List[str]]:
    """
    Complete clustering pipeline.
    """
    embeddings = embed_terms(terms, model_name=model_name)

    k_list, scores = evaluate_k_range(embeddings, k_list)
    optimal_k = find_optimal_k(k_list, scores)
    print(optimal_k)
    

    plt.figure(figsize=(8, 5))
    plt.plot(k_list, scores, marker="o", linewidth=2)
    plt.axvline(
        x=optimal_k,
        color="red",
        linestyle="--",
        linewidth=2,
        label=f"Optimal k = {optimal_k}"
    )
    plt.xlabel("Number of clusters (k)")
    plt.ylabel("Silhouette score")
    plt.title("K evaluation curve")
    plt.grid(alpha=0.3)
    plt.legend()
    os.makedirs("figures", exist_ok=True)
    plt.savefig(os.path.join("figures", f"{figure_name}.jpg"), dpi=300)
    plt.show()

    print(f"Optimal k selected: {optimal_k}")

    final_clusters = generate_final_clusters(
        terms,
        embeddings,
        optimal_k
    )

    return final_clusters


In [5]:
def get_top_k_similar(embedding, embeddings, names, k=5):
    embedding = embedding.reshape(1, -1)
    similarities = cosine_similarity(embedding, embeddings)[0]
    top_k_indices = np.argsort(similarities)[::-1][:k]
    top_k_names = [names[i] for i in top_k_indices]
    top_k_scores = [similarities[i] for i in top_k_indices]
    return top_k_names, top_k_scores

In [6]:
from sentence_transformers import SentenceTransformer, util
from typing import List
import numpy as np


def generate_embeddings_in_batches(
    texts,
    batch_size= 1024,
    embedder=SentenceTransformer('all-mpnet-base-v2',device="cuda:1")
):
    """
    Generate sentence embeddings in batches.
    
    Args:
        texts (List[str]): List of input text strings.
        model_name (str): Pretrained SentenceTransformer model name.
        batch_size (int): Number of texts to process per batch.
    
    Returns:
        np.ndarray: Array of embeddings.
    """
    embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        batch_embeddings = embedder.encode(batch, convert_to_numpy=True)
        embeddings.append(batch_embeddings)

    return np.vstack(embeddings)


In [7]:
from pydantic import BaseModel, validator
from typing import Dict, List
from tqdm import tqdm

openai_client = openai.OpenAI(api_key=openai.api_key)
# Corrected model for GPT response
class GPTClusterResponse(BaseModel):
    label: str
    subtopics: List[List[str]]

    @validator('label')
    def validate_label(cls, v):
        v = v.strip().lower()
        if v not in {"yes", "no"}:
            raise ValueError("label must be 'yes' or 'no'")
        return v

    @validator('subtopics')
    def validate_subtopics(cls, v, values):
        label = values.get('label')
        if label == "yes":
            # "yes" means coherent topic -> no subtopic split expected
            if v is None or (isinstance(v, list) and len(v) == 0):
                return []
            raise ValueError("If label is 'yes', subtopics must be an empty list or omitted")
        return v

def generate_gpt4_structured_response(content, print_output=False):
    response = openai_client.responses.parse(
        model="gpt-4o",
        input=[
            {
                "role": "user",
                "content": content,
            },
        ],
        text_format=GPTClusterResponse,
    )

    parsed_response = response.output_parsed

    if print_output:
        print(parsed_response)

    # Extract and return the first response
    return parsed_response

# Main clustering function
def break_cluster(cluster_dict: Dict[str, List[str]]) -> Dict[str, List[List[str]]]:
    cluster_dict_cleaned = {}

    for cid in tqdm(sorted(cluster_dict.keys())):
        keywords = cluster_dict[cid]
        prompt = f"""
{keywords}
Please help determine if these terms focus on a coherent topic or not. Answer strictly in the following JSON format:

If the terms form a coherent topic:
{{
  "label": "yes",
  "subtopics": []
}}

If not:
{{
  "label": "no",
  "subtopics": [
    [list of keywords for topic 1],
    [list of keywords for topic 2],
    ...
  ]
}}
**Important**: If the answer is no, group the terms by subtopics. Do not add new terms or remove existing terms from the provided list of terms. 
"""

        # parsed = generate_gpt4_structured_response(prompt)

        try:
            parsed = generate_gpt4_structured_response(prompt)
            # print(type(parsed))
            # print(parsed)

            if parsed.label == "yes":
                cluster_dict_cleaned[cid] = [keywords]
            elif parsed.label == "no":
                # for i, sublist in enumerate(parsed.subtopics):
                cluster_dict_cleaned[cid] = parsed.subtopics

        except Exception as e:
            print(f"[Validation error] Cluster '{cid}': {e}")
            cluster_dict_cleaned[cid] = [keywords]
            continue

    return cluster_dict_cleaned


def break_cluster_postprocess(dict_original_clusters, dict_subclusters):
    dict_cleaned_subclusters = defaultdict(list)
    topic_ids = dict_original_clusters.keys()
    embedder=SentenceTransformer('all-mpnet-base-v2',device="cuda:1")
    for idx in tqdm(topic_ids):
        keywords = dict_original_clusters[idx]
        # if idx not in dict_subclusters.keys():
        #     subclusters = [keywords]
        # else:
        subclusters = dict_subclusters[idx]

        seen = set()
        unique_subclusters = []
        for sub in subclusters:
            norm = tuple(sorted(sub))
            if norm not in seen:
                seen.add(norm)
                unique_subclusters.append(sub)
        
        subcluster_keywords = [keyword for keywords in unique_subclusters for keyword in keywords]

        # If GPT returned empty/invalid subclusters, keep original keywords as one subcluster.
        if len(subcluster_keywords) == 0:
            print(idx, keywords, "no subcluster")
            dict_cleaned_subclusters[idx] = [list(set(keywords))] if len(keywords) > 0 else []
            continue
    

        #Find terms that have not been included
        keyword_not_included = []
        for keyword in keywords:
            if keyword not in subcluster_keywords:
                keyword_not_included.append(keyword)


        
        # assign terms to subclusters
        subcluster_keywords_embedding = generate_embeddings_in_batches(subcluster_keywords, batch_size=1, embedder=embedder)

        for keyword in keyword_not_included:
            original_keywords_embedding = generate_embeddings_in_batches([keyword], batch_size=1, embedder=embedder)[0, :]
            top_names, top_scores = get_top_k_similar(original_keywords_embedding, subcluster_keywords_embedding, subcluster_keywords, k=1)
            # print(keyword, top_names, top_scores)
            most_similar_name = top_names[0]

            for j, s in enumerate(unique_subclusters):
                if most_similar_name in s:
                    unique_subclusters[j].append(keyword)
                   
                    
        #Remove terms that do not exist
        filtered_subclusters = [[item for item in sublist if item in keywords] for sublist in unique_subclusters]
        filtered_subclusters = [sublist for sublist in filtered_subclusters if sublist]
        filtered_subclusters = [list(set(sublist)) for sublist in filtered_subclusters]

        seen = set()
        unique_filtered_subclusters = []
        for sub in filtered_subclusters:
            norm = tuple(sorted(sub))
            if norm not in seen:
                seen.add(norm)
                unique_filtered_subclusters.append(sub)
            
        dict_cleaned_subclusters[idx] = unique_filtered_subclusters
        
    embedder = embedder.cpu()      # move weights off GPU
    del embedder                # remove reference
    # del embeddings         # only if you no longer need them
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    return dict_cleaned_subclusters
        

In [8]:
def topic_naming(dict_topic_keyword, domain, similarity_threshold=0, k=3):

    embeddings = np.empty((0, 768))  # Adjust dimension to your embedding size
    names = []
    dict_topic_id2name = {}
    embedder=SentenceTransformer('all-mpnet-base-v2',device="cuda:1")
    for tid, keyword_distribution in tqdm(dict_topic_keyword.items()):

        name = generate_name(keyword_distribution, domain)
        
        embedding = generate_embeddings_in_batches([name], batch_size=1, embedder=embedder)[0, :]  # shape (768,)
        if name in names:
            dict_topic_id2name[tid] = name
            continue
        if len(names) == 0:
            names.append(name)
            embeddings = np.vstack([embeddings, embedding])
            dict_topic_id2name[tid] = name
        else:
            top_k_names, top_k_scores = get_top_k_similar(embedding, embeddings, names, k=k)
            # print(top_k_names, top_k_scores)
            
            if top_k_scores[0] < similarity_threshold:  # Check if it's dissimilar enough to be new
                names.append(name)
                embeddings = np.vstack([embeddings, embedding])
                dict_topic_id2name[tid] = name
            else:
                selected_name = check_existing_names_suitability(name, top_k_names, domain)
                # print('selected_name', selected_name)
                if selected_name not in names:
                    names.append(selected_name)
                    embeddings = np.vstack([embeddings, embedding])
                dict_topic_id2name[tid] = selected_name
            if 'machine learning' in keyword_distribution.keys():
                print(name)
                print(top_k_names)
                print(selected_name)
    embedder = embedder.cpu()      # move weights off GPU
    del embedder                # remove reference
    # del embeddings         # only if you no longer need them
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    return dict_topic_id2name
    
def generate_name(keyword_distribution, domain='method',max_retries=5):

    if len(keyword_distribution)==1:
        name = list(keyword_distribution.keys())[0]

    if domain=='method':
        prompt = f"""
You are an expert in the taxonomy construction.
Please use 1 precise and succinct phrase in no more than 3 words to summarize the main topic of the following list of keywords in the methodology domain based on their distribution:
{keyword_distribution}
"""
    else:
        prompt = f"""
You are an expert in the taxonomy construction.
Please use 1 precise and succinct phrase in no more than 3 words to summarize the main topic of the following list of keywords in the health domain based on their distribution:
{keyword_distribution}
"""
    for attempt in range(max_retries):  
        try:
            name = generate_gpt4_response(prompt).rstrip('.')
            return name
        except Exception as e:
            print(f"Error generating name: {e}")
            continue
    if attempt == max_retries - 1:
        name = list(keyword_distribution.keys())[0]
        print(f"Failed to generate name after {max_retries} attempts. Using {name} as a fallback.")
    return name

def check_existing_names_suitability(name, similar_names, domain, max_retries=5):    
    phrase_str = ''
    for idx, s_name in enumerate(similar_names):
        phrase_str += f'{idx}. {s_name}\n'

    if domain == 'method':
        prompt = f"""
        Please check if any of these phrases is equivalent to '{name}'.
        Here is the list of phrases:
        {phrase_str}
    
        Return the ID only of the top 1 suitable phrase without providing the rationale. If none, return none only. 
        """
    else:
        prompt = f"""
        Please check if any of these phrases is equivalent to '{name}'.
        Here is the list of phrases:
        {phrase_str}
    
        Return the ID only of the top 1 suitable phrase without providing the rationale. If none, return none only. 
        """
    for attempt in range(max_retries):
        response = generate_gpt4_response(prompt).strip().rstrip('.').lower()

        if response[:4] == 'none':
            return name
        elif response[0].isdigit():
            idx = int(response[0])
            if 0 <= idx < len(similar_names):
                return similar_names[idx]
        
        # Retry on invalid response
        print(f"Invalid response '{response}', retrying ({attempt+1}/{max_retries})...")

    # Fallback if no valid response is received
    return name
    
def generate_gpt4_response(content, print_output=False):
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "user", "content": content}
        ],
        temperature=0,
        top_p=0.1,
        n=1,
        # max_tokens=2000,  # Uncomment if you want to limit output length
        # stop=["###"],    # Uncomment if you want to specify stop tokens
    )

    if print_output:
        print(response)

    # Extract and return the first response
    return response.choices[0].message.content


In [9]:
def build_topic_keyword_counts(keyword_to_pmids, topic_dict):
    
    topic_terms = {}
    for cid, keyword_list in topic_dict.items():
        for idx, keywords in enumerate(keyword_list):
            topic_terms[f"{cid}-{idx}"] = {kw: 0 for kw in set(keywords)}
    
    
    for t_id, keywords in topic_terms.items():
        for keyword in keywords:
            if keyword in keyword_to_pmids:
                topic_terms[t_id][keyword] = len(keyword_to_pmids[keyword])
            else:
                topic_terms[t_id][keyword] = 0
                print(keyword, "not found in keyword_to_pmids")
                
        
    
    return topic_terms


def convert_counts_to_proportions(topic_keyword_count):
    topic_keyword_proportion = {}

    for topic_id, keywords in topic_keyword_count.items():
        total = sum(keywords.values())
        if total == 0:
            topic_keyword_proportion[topic_id] = {kw: 0.0 for kw in keywords}
        else:
            topic_keyword_proportion[topic_id] = {
                kw: count / total for kw, count in keywords.items()
            }

    return topic_keyword_proportion

In [10]:
def combine_topic_w_same_name(original_dict):
    from nltk.stem import PorterStemmer
    from collections import defaultdict
    stemmer = PorterStemmer()

    unique_values = defaultdict(list)
    for k, v in original_dict.items():
        stemmed = ' '.join([stemmer.stem(word.lower()) for word in v.split()])
        unique_values[stemmed].append(k)
            

    return unique_values
    

In [11]:
def dict_topic_id_to_topic_name(topic_stemmed_names,  topic_names):
    id2name = {}
    name2ids = {}
    for stemmed_name, id_list in topic_stemmed_names.items():
        for i in id_list:
            id2name[i] = topic_names[id_list[0]]
        name2ids[topic_names[id_list[0]]] = id_list
        
    return id2name, name2ids



In [12]:
def compute_pairwise_cosine_similarity(topic_names):
    embedder=SentenceTransformer('all-mpnet-base-v2',device="cuda:1")
    embeddings = generate_embeddings_in_batches(topic_names, batch_size=1, embedder=embedder)
    # Compute full cosine similarity matrix
    print('computing cosine similarity...')
    similarity_matrix = cosine_similarity(embeddings)
    
    # Create dictionary of pairwise similarities (i < j to avoid duplicates)
    similarity_dict = {}
    n = similarity_matrix.shape[0]
    for i in range(n):
        for j in range(i + 1, n):
            similarity_dict[(topic_names[i], topic_names[j])] = similarity_matrix[i, j]
    
    embedder = embedder.cpu()      # move weights off GPU
    del embedder                # remove reference
    # del embeddings         # only if you no longer need them
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    return similarity_dict


In [13]:
def find_threshold_by_pct(topic_dict, n_subset_topic1 = 200, precision=1e-3):

    # Step 1: Group by topic1
    topic_groups = defaultdict(list)
    for (t1, t2), score in topic_dict.items():
        topic_groups[t1].append((t2, score))

    
    all_topic1 = list(topic_groups.keys())
    subset_topic_groups = {t1: topic_groups[t1] for t1 in all_topic1[:n_subset_topic1]}

    
    thresholds = {}

    # Step 2 & 3: Binary search over pct [0.0, 1.0]
    for topic1, topic2_scores in tqdm(subset_topic_groups.items()):
        sorted_topic2s = sorted(topic2_scores, key=lambda x: x[1], reverse=False)
        # Extract just topic2 names
        sorted_topic2_names = [t2 for t2, _ in sorted_topic2s]
        sorted_topic2_scores = [s for _, s in sorted_topic2s]
        
        n = len(sorted_topic2_names)
        low, high = 0, 1.0
        best_pct = 0

        # has_relation, prev_has_relation = None, None
        prev_topic2 = None
        while high - low > precision:
            mid_pct = (low + high) / 2
            k = max(1, int(mid_pct * n))  # at least one
            selected_topic2 = sorted_topic2_names[k - 1]
            selected_score2 = sorted_topic2_scores[k-1]

            has_relation = have_relationship(topic1, selected_topic2)
            if has_relation:
                best_pct = mid_pct
                high = mid_pct
            else:
                low = mid_pct
            
            if prev_topic2 == selected_topic2:
                break
            else:
                prev_topic2 = selected_topic2                

        
        thresholds[topic1] = best_pct

    return thresholds

def have_relationship(topic1, topic2, max_attempts=3):
    for attempt in range(max_attempts):
        try:
            prompt = (
            f"""
            Determine the relationship between Topic A and Topic  B. Use the following relationship labels:
            Superset: Topic B is a subcategory of Topic A, but Topic A is not a subcategory of Topic B.
            Subset: Topic A is a subcategory of Topic B, but Topic B is not a subcategory of Topic A.
            Equal: Topic A is a Topic B and Topic B is a Topic A.
            NoOverlap: The two topics are conceptually distinct.
            
            Here are the topics:
            Topic A: {topic1}\nTopic B: {topic2}
            
            Return the label only.
            """
            )

            response = generate_gpt4_response(prompt)
            # print(response)
            if response.strip(".").lower()!='nooverlap':
                return True
            else:
                return False
        except Exception as e:
            print(f"Error in have_relationship: {e}")
            continue
    return False
    
    

In [ ]:
def check_topic_relation(topic_dict, topic_prop_thresholds, lowest_threshold):
    # Step 1: Group by topic1 
    topic_groups = defaultdict(list)
    for (t1, t2), score in topic_dict.items():
        topic_groups[t1].append((t2, score))
        
    
    #Step 2:  select their related topic2 with high similarities and check relations
    dict_topic_relation = {}
    for topic1, topic2_scores in tqdm(topic_groups.items()):
        sorted_topic2s = sorted(topic2_scores, key=lambda x: x[1], reverse=True)
        n = len(sorted_topic2s)
        # Do not need to check such topic 1 since it's not related to any other topic.
        if topic_prop_thresholds[topic1] < lowest_threshold:
            continue
        k = max(1, int((1-topic_prop_thresholds[topic1]) * n))
        selected_topic2s = sorted_topic2s[:k]
        for topic2, sim in selected_topic2s:
            relation = check_relationship(topic1, topic2)
            dict_topic_relation[(topic1, topic2)] = relation

        
    return dict_topic_relation

def check_relationship(topic1, topic2, max_retries=5, retry_delay=1):
    prompt = (
        f"""
        Determine the relationship between Topic A and Topic  B. Use the following relationship labels:
        Superset: Topic B is a subcategory of Topic A, but Topic A is not a subcategory of Topic B.
        Subset: Topic A is a subcategory of Topic B, but Topic B is not a subcategory of Topic A.
        Equal: Topic A is a Topic B and Topic B is a Topic A.
        NoOverlap: The two topics are conceptually distinct.
        
        Here are the topics:
        Topic A: {topic1}\nTopic B: {topic2}
        
        Return the label only.
        """
    )
    for attempt in range(max_retries):
        try:
            response = generate_gpt4_response(prompt)
            if response:
                return response.strip(".").lower()
        except Exception as e:
            # Optional: log the exception e
            print(f"Error in check_relationship: {e}, {topic1}, {topic2}")
            pass
        time.sleep(retry_delay)  # Wait before retrying
    
    # If all attempts fail, return a default
    return "nooverlap"


In [15]:
import networkx as nx

def merge_equivalent_concepts(dict_topic_relation):
    """
    Merge equivalent concepts using a NetworkX graph.

    Args:
        dict_topic_relation (dict): key = (topic1, topic2), value = relation

    Returns:
        shown_to_originals (dict): key = shown name, value = list of original concepts
        original_to_shown (dict): key = original concept, value = shown name
    """
    dict_relation_count = Counter()
    pairs = []
    for topics, relation in dict_topic_relation.items():
        dict_relation_count[relation] += 1
        if relation == 'equal':
            print(topics)
            pairs.append(topics)
            
    print("relation count:", dict_relation_count)
    # Build graph
    G = nx.Graph()
    G.add_edges_from(pairs)

    # Also ensure isolated nodes appear if needed
    for c1, c2 in pairs:
        G.add_node(c1)
        G.add_node(c2)

    def word_count(s):
        return len(s.split())

    shown_to_originals = {}
    original_to_shown = {}

    # Each connected component = one equivalence group
    for component in nx.connected_components(G):
        concepts = list(component)

        # Choose shortest by word count, then char length, then alphabetically
        shown = min(concepts, key=lambda s: (word_count(s), len(s), s))

        shown_to_originals[shown] = concepts

        for c in concepts:
            original_to_shown[c] = shown
    

    return shown_to_originals, original_to_shown

def merge_equal_topics(dict_topic_relation, original_to_shown):
    new_map = {}
    for topic_pair, rel in dict_topic_relation.items():
        t1, t2 = topic_pair
        t1 = original_to_shown.get(t1, t1)
        t2 = original_to_shown.get(t2, t2)
        new_map[(t1, t2)] = rel
    return new_map

def extract_subset_relation(merged_topic_relation):
    subset_relation = []
    for pair, rel in merged_topic_relation.items():
        if rel == 'equal' or rel == 'nooverlap':
            continue
        t1, t2 = pair
        if rel == 'superset':
            t1, t2 = t2, t1
        subset_relation.append((t1, t2))
    return subset_relation

In [16]:
def double_check_subsumption_relation(edges, reverse):
    dict_reverse_relation = {}
    for topic1, topic2 in tqdm(edges):
        if reverse:
            relation = check_relationship(topic2, topic1)
        else:
            relation = check_relationship(topic1, topic2)
        dict_reverse_relation[(topic1, topic2)] = relation
    return dict_reverse_relation

def select_consistent_edge_subset(check_nth_relation, reverse):
    dict_relation_count = Counter()
    subset_edges = []
    for topics, relation in check_nth_relation.items():
        dict_relation_count[relation] += 1
        if relation == ('superset' if reverse else 'subset'):
            # print(topics)
            subset_edges.append(topics)
    print(dict_relation_count)
    return subset_edges

In [17]:
import ast

def fix_cycles_collect_unresolved(
    edges,
    retry_attempts=5,
    verbose=False
):
    """
    Loop:
      - find a cycle not in skip set,
      - ask model up to retry_attempts times for edges to remove,
      - if it yields existing edges, remove them and continue,
      - else record the cycle as unresolved (for manual review), add to skip set, and continue.

    Stops when either:
      - the graph is acyclic, or
      - all remaining cycles are in the unresolved (skip) set.

    Returns:
      clean_edges, removed_edges, unresolved_cycles (list), rounds
    """
    edges_list = [tuple(e) for e in edges]     # normalize to (u,v)
    removed = []
    unresolved = []
    skip_cycles = set()
    rounds = 0

    while True:
        cycle = find_next_cycle(edges_list, skip_cycles=skip_cycles)
        if cycle is None:
            if verbose:
                if unresolved:
                    print("[done] no more resolvable cycles (only unresolved remain).")
                else:
                    print("[done] acyclic.")
            return edges_list, removed, unresolved, rounds

        rounds += 1
        if verbose:
            print(f"[round {rounds}] cycle: {cycle}")

        edges_set = set(edges_list)
        to_remove = []
        for attempt in range(1, retry_attempts + 1):
            wrong_edges = break_a_cycle(cycle) or []
            # keep only edges that actually exist in the graph
            to_remove = [tuple(e) for e in wrong_edges if tuple(e) in edges_set]
            if to_remove:
                if verbose:
                    print(f"[round {rounds}] attempt {attempt}: removing {to_remove}")
                break
            if verbose:
                print(f"[round {rounds}] attempt {attempt}: no valid edges from model")

        if not to_remove:
            # Couldn’t resolve this cycle now; record and skip it in future searches
            unresolved.append(cycle)
            skip_cycles.add(cycle)
            if verbose:
                print(f"[blocked] stored unresolved cycle and moving on: {cycle}")
            # Do NOT modify edges; just continue to look for other cycles
            continue

        # Apply removals and continue
        edges_set.difference_update(to_remove)
        edges_list = list(edges_set)
        removed.extend(to_remove)

        if verbose:
            print(f"[round {rounds}] removed: {to_remove}")

def find_next_cycle(edges, skip_cycles=None):
    """
    Find one directed cycle (as a CLOSED normalized tuple).
    If the first found cycle is in skip_cycles, try to find a different cycle by
    temporarily hiding one edge at a time from that cycle (detection-only).
    Return None if no cycle exists or only skipped cycles remain.
    """
    skip_cycles = skip_cycles or set()

    G = nx.DiGraph()
    G.add_edges_from(edges)

    
    def find_cycle_in_graph(graph: nx.DiGraph):
        """
        Find one directed cycle and return it as a CLOSED, normalized tuple
        (n1, n2, ..., nk, n1). Return None if no cycle exists.
        """
        try:
            cyc_edges = nx.find_cycle(graph, orientation="original")
            # Convert edge list to node cycle [u1, u2, ..., uk, u1]
            cycle = [cyc_edges[0][0]] + [v for (_, v, *_) in cyc_edges]
            return _normalize_closed_cycle(cycle)
        except nx.NetworkXNoCycle:
            return None
    # First attempt
    cyc = find_cycle_in_graph(G)
    if cyc is None:
        return None
    if cyc not in skip_cycles:
        return cyc

    # Try to uncover a different cycle by temporarily removing each edge of the skipped cycle
    path_edges = list(zip(cyc[:-1], cyc[1:]))  # [(n1,n2),...,(nk,n1)]
    for e in path_edges:
        if not G.has_edge(*e):
            continue
        G.remove_edge(*e)
        alt = find_cycle_in_graph(G)
        G.add_edge(*e)  # restore
        if alt is not None and alt not in skip_cycles:
            return alt

    # No alternative cycles found (besides already skipped)
    return None

def _normalize_closed_cycle(cycle):
    """
    Given a CLOSED cycle (last node repeats the first), rotate so the smallest node
    is first, and return a CLOSED tuple.
    Example: ('C','A','B','C') -> ('A','B','C','A')
    """
    assert cycle[0] == cycle[-1], "cycle must be CLOSED (last == first)"
    core = cycle[:-1]  # exclude the repeated last
    min_idx = min(range(len(core)), key=lambda i: core[i])  # natural ordering
    rotated = core[min_idx:] + core[:min_idx]
    return tuple(rotated + [rotated[0]])



def break_a_cycle(cycle, max_retries=5, retry_delay=1):
    prompt = (
        f"""
In the following cycle, each item is a subcategory of the item that directly follows it. For example, in (A, B, C, A), A is a subcategory of B, B of C, and so on.
A subcategory means "A is a narrower, more specific task or concept than B."
Cycle: {cycle}
Please identify invalid child-parent relationships in the format [('A','B'), ('B','C')] and do not provide any rationale.
        """
    )
    for attempt in range(max_retries):
        try:
            response = generate_gpt4_response(prompt)
            if response:
                pattern = r"\[\s*(\('.*?'\s*,\s*'.*?'\)\s*(,\s*\('.*?'\s*,\s*'.*?'\)\s*)*)\]"
                
                m = re.search(pattern, response, flags=re.S)
                if m:
                    tuple_list = ast.literal_eval(m.group())
                    print(tuple_list)

                    return tuple_list
        except Exception as e:
            # Optional: log the exception e
            pass
        time.sleep(retry_delay)  # Wait before retrying
    return []


In [18]:
from collections import defaultdict, deque

def remove_transitive_edges(edges, verbose=True):
    """
    Remove transitive edges from a taxonomy DAG.
    Edges are assumed to be (child, parent).

    Prints removed edges and the reason.
    """

    # Build adjacency: child -> parents
    graph = defaultdict(set)
    for child, parent in edges:
        graph[child].add(parent)

    def find_alternate_path(start, end):
        """
        Find an alternate path start -> ... -> end (length >= 2).
        Returns the path if found, else None.
        """
        queue = deque([(start, [start])])
        visited = {start}

        while queue:
            node, path = queue.popleft()
            for parent in graph[node]:
                if parent == end:
                    return path + [end]
                if parent not in visited:
                    visited.add(parent)
                    queue.append((parent, path + [parent]))
        return None

    reduced = []

    for child, parent in edges:
        # Temporarily remove direct edge
        graph[child].remove(parent)

        path = find_alternate_path(child, parent)

        if path:
            if verbose:
                print(
                    f"Removed transitive edge: ({child} → {parent}) "
                    f"because path exists: {' → '.join(path)}"
                )
        else:
            reduced.append((child, parent))

        # Restore edge
        graph[child].add(parent)

    return reduced



In [19]:
def get_sub_hierarchies_root_nodes(edges):
    """
    Identify root nodes in hierarchies:
    - Root nodes are parents that are never children.
    - Also includes isolated nodes (not connected in any edge).
    """
    # Unpack children and parents from edges
    children = {child for child, _ in edges}
    parents = {parent for _, parent in edges}

    # Root nodes: nodes that are parents but never children
    root_nodes = parents - children

    connected_nodes = {n for edge in edges for n in edge}
    
    nonroot_nodes = connected_nodes - root_nodes

    print(len(root_nodes), len(nonroot_nodes))
    return root_nodes, nonroot_nodes


In [20]:
def check_high_level_topic_relation(dict_topic_whole, topic_prop_thresholds,dict_topic_selected):

    #Extract the similarity score threshold for each topic 1.
    topic_sim_score_thresholds = {}
    topic_groups_whole = defaultdict(list)
    for (t1, t2), score in dict_topic_whole.items():
        topic_groups_whole[t1].append((t2, score))
    for topic1, topic2_scores in topic_groups_whole.items():
        sorted_topic2s = sorted(topic2_scores, key=lambda x: x[1], reverse=True)
        n = len(sorted_topic2s)
        idx = max(1, int((1-topic_prop_thresholds[topic1]) * n))-1
        sim_score_threshold = sorted_topic2s[idx][1]
        topic_sim_score_thresholds[topic1] = sim_score_threshold
    
    topic_groups = defaultdict(list)
    for (t1, t2), score in dict_topic_selected.items():
        topic_groups[t1].append((t2, score))
    
    selected_topic_pairs = []
    for topic1, topic2_scores in tqdm(topic_groups.items()):
        if topic1 not in topic_sim_score_thresholds:
            sim_score_threshold = 0
        else:
            sim_score_threshold = topic_sim_score_thresholds[topic1]
        for topic2, score in topic2_scores:
            if score >= sim_score_threshold:
                selected_topic_pairs.append((topic1, topic2))
    
    dict_topic_relation = {}     
    for topic1, topic2 in tqdm(selected_topic_pairs):
        relation = check_relationship(topic1, topic2)
        dict_topic_relation[(topic1, topic2)] = relation
    return dict_topic_relation
        

In [21]:
from collections import defaultdict

def write_hierarchy_to_file(edges, filename):
    # Step 1: Create parent -> children map
    tree = defaultdict(list)
    all_children = set()

    for child, parent in edges:
        tree[parent].append(child)
        all_children.add(child)

    # Step 2: Identify root nodes (nodes that are never children)
    roots = [node for node in tree if node not in all_children]
    print("number of roots:", len(roots))

    # Step 3: Recursive function to write the tree
    def write_tree_to_file(file, node, level=0):
        file.write('\t' * level + f"- {node}\n")
        for child in tree.get(node, []):
            write_tree_to_file(file, child, level + 1)

    # Step 4: Write the tree to a file
    with open(filename, 'w', encoding='utf-8') as f:
        for root in roots:
            write_tree_to_file(f, root)

    print(f"Hierarchy tree has been written to '{filename}'")

In [ ]:
# Domain switch: M_group = methodological innovation; H_group = health.
# All cells below use this domain — change input/output paths (_m ↔ _h) accordingly.
keywords = load_from_pickle("data/processed/M_group.pkl")

In [47]:
clusters = cluster_terms_pipeline(keywords, k_list=np.arange(0, 2000, 50)[1:], figure_name="k_evaluation_curve_M_group", model_name="all-mpnet-base-v2")

save_to_pickle(clusters, "data/processed/clusters_m.pkl")

In [ ]:
cluster_dict_subclusters = break_cluster(clusters)
save_to_pickle(cluster_dict_subclusters, 'data/processed/cluster_dict_subclusters_m.pkl')
#Note that a keyword may be assigned to multiple subclusters of a cluster by GPT.

In [ ]:
cluster_dict_subclusters_cleaned = break_cluster_postprocess(clusters, cluster_dict_subclusters)
save_to_pickle(cluster_dict_subclusters_cleaned, 'data/processed/cluster_dict_subclusters_cleaned_m.pkl')


In [22]:
cluster_dict_subclusters_cleaned = load_from_pickle('data/processed/cluster_dict_subclusters_cleaned_m.pkl')


In [23]:
keywords_to_pmids = load_from_pickle('data/processed/M_keyword_to_pmids.pkl')
dict_topic_keyword_proportion = convert_counts_to_proportions(build_topic_keyword_counts(keywords_to_pmids, cluster_dict_subclusters_cleaned))


In [24]:
topic_names = topic_naming(dict_topic_keyword_proportion, domain='method', similarity_threshold=0, k=3)
save_to_pickle(topic_names, 'data/processed/topic_names_m.pkl')


In [25]:
print("number of subclusters:", len(dict_topic_keyword_proportion))

In [26]:
topic_names_cleaned = combine_topic_w_same_name(topic_names)
print("number of topics after combining:", len(topic_names_cleaned))

In [27]:
id2name, name2ids = dict_topic_id_to_topic_name(topic_names_cleaned,  topic_names)

save_to_pickle(id2name, 'data/processed/id2name_m.pkl')
save_to_pickle(name2ids, 'data/processed/name2ids_m.pkl')

In [28]:
topic_name_list = list(name2ids.keys())
print(len(topic_name_list))
pairwise_topic_similarity = compute_pairwise_cosine_similarity(
    topic_name_list)
len(pairwise_topic_similarity)

In [29]:
save_to_pickle(pairwise_topic_similarity, 'data/processed/pairwise_topic_similarity_m.pkl')

In [37]:
dict_topic1_pro_threshold = find_threshold_by_pct(pairwise_topic_similarity, n_subset_topic1=len(topic_name_list), precision=0.005)
save_to_pickle(dict_topic1_pro_threshold, 'data/processed/dict_topic1_pro_threshold_m.pkl')

In [ ]:
set(topic_name_list) - set(dict_topic1_pro_threshold.keys()) #1029->1028 is because of upper triangle

In [39]:
thresholds = list(dict_topic1_pro_threshold.values())
print(np.min(thresholds), np.median(thresholds), np.average(thresholds), np.max(thresholds), len(thresholds))

import matplotlib.pyplot as plt
plt.figure(figsize=(4,3))
plt.hist(thresholds, bins=20, edgecolor='black')
plt.grid(True)
plt.tight_layout()
plt.show()

In [40]:
print(len([t for t in thresholds if t<0.9]), len([t for t in thresholds if t>=0.9]))
print(len([t for t in thresholds if t<0.95]), len([t for t in thresholds if t>=0.95]))


In [ ]:
lowest_threshold = 0.9
dict_topic_relation = check_topic_relation(pairwise_topic_similarity, dict_topic1_pro_threshold, lowest_threshold=lowest_threshold)

save_to_pickle(dict_topic_relation, f'data/processed/dict_topic_relation_m_{lowest_threshold}.pkl')

In [ ]:
lowest_threshold = 0.9
dict_topic_relation = load_from_pickle(f'data/processed/dict_topic_relation_m_{lowest_threshold}.pkl')
shown_to_originals_low_level, original_to_shown_low_level = merge_equivalent_concepts(dict_topic_relation)
print(len(dict_topic_relation))
merged_topic_relation = merge_equal_topics(dict_topic_relation, original_to_shown_low_level)
print(len(merged_topic_relation))
subset_edges_low_level = extract_subset_relation(merged_topic_relation)
print(len(subset_edges_low_level))
save_to_pickle(subset_edges_low_level, 'data/processed/subset_edges_low_level_m_0.pkl')




In [ ]:
save_to_pickle(original_to_shown_low_level, 'data/processed/original_to_shown_low_level_m.pkl')
save_to_pickle(shown_to_originals_low_level, 'data/processed/shown_to_originals_low_level_m.pkl')



In [ ]:
subset_edges_low_level_updated = subset_edges_low_level

reverse = True
for i in np.arange(1, 6, 1):
    print(i)
    dict_topic_relation_updated = double_check_subsumption_relation(subset_edges_low_level_updated, reverse=reverse)
    save_to_pickle(dict_topic_relation_updated, f'data/processed/dict_topic_relation_updated_m_{i}.pkl')
    subset_edges_low_level_updated = select_consistent_edge_subset(dict_topic_relation_updated, reverse=reverse)
    save_to_pickle(subset_edges_low_level_updated, f'data/processed/subset_edges_low_level_m_{i}.pkl')
    reverse = not reverse



In [43]:
subset_edges_low_level_num_all_iters = []
for i in np.arange(0, 6, 1):
    subset_edges_low_level = load_from_pickle(f'data/processed/subset_edges_low_level_m_{i}.pkl')
    subset_edges_low_level_num_all_iters.append(len(subset_edges_low_level))

plt.figure(figsize=(4,3))
plt.plot(np.arange(0, 6, 1), subset_edges_low_level_num_all_iters, marker='o', linestyle='-', color='b')
plt.xlabel('Iteration')
plt.ylabel('Number of Subset Edges')
plt.title('Subset Edges Number Over Iterations')
plt.grid(True)



In [44]:
selected_iter = 2
subset_edges_low_level_selected = load_from_pickle(f'data/processed/subset_edges_low_level_m_{selected_iter}.pkl')




In [45]:
subset_edges_low_level_cycle_removed, removed_edges, unresolved_cycles, rounds = (
    fix_cycles_collect_unresolved(subset_edges_low_level_selected, verbose=True)
)


print(f"edges before cycle removal:",
        len(subset_edges_low_level_selected))
print(f"edges after cycle removal:",
        len(subset_edges_low_level_cycle_removed))
print(f"unresolved cycles:",
        unresolved_cycles)


In [46]:
subset_edges_low_level_direct = remove_transitive_edges(subset_edges_low_level_cycle_removed)

save_to_pickle(
        subset_edges_low_level_direct,
        "data/processed/subset_edges_low_level_direct_m.pkl")



In [47]:
sub_roots_low_level, nonroot_nodes_low_level = get_sub_hierarchies_root_nodes(subset_edges_low_level_direct)

topic_name_list_updated = list(set([original_to_shown_low_level.get(node, node) for node in topic_name_list]))
print(len(topic_name_list), len(topic_name_list_updated))
nodes_high_level = list(set(topic_name_list_updated) - set(nonroot_nodes_low_level))
print(len(nodes_high_level))

In [48]:
pairwise_topic_similarity_high_level = compute_pairwise_cosine_similarity(nodes_high_level)
print(len(pairwise_topic_similarity_high_level))


In [49]:
save_to_pickle(pairwise_topic_similarity_high_level, 'data/processed/pairwise_topic_similarity_high_level_m.pkl')

In [ ]:
dict_high_level_topic_relation = check_high_level_topic_relation(pairwise_topic_similarity, dict_topic1_pro_threshold, pairwise_topic_similarity_high_level)
save_to_pickle(dict_high_level_topic_relation, f'data/processed/dict_high_level_topic_relation_m.pkl')



In [ ]:
shown_to_originals_high_level, original_to_shown_high_level = merge_equivalent_concepts(dict_high_level_topic_relation)
print(len(dict_high_level_topic_relation))
merged_topic_relation_high_level = merge_equal_topics(dict_high_level_topic_relation, original_to_shown_high_level)
print(len(merged_topic_relation_high_level))
subset_edges_high_level = extract_subset_relation(merged_topic_relation_high_level)
print(len(subset_edges_high_level))

save_to_pickle(subset_edges_high_level, 'data/processed/subset_edges_high_level_m_0.pkl')


In [ ]:
subset_edges_high_level_updated = subset_edges_high_level

reverse = True
for i in np.arange(1, 6, 1):
    print(i)
    dict_topic_relation_high_level_updated = double_check_subsumption_relation(subset_edges_high_level_updated, reverse=reverse)
    save_to_pickle(dict_topic_relation_high_level_updated, f'data/processed/dict_topic_relation_high_level_updated_m_{i}.pkl')
    subset_edges_high_level_updated = select_consistent_edge_subset(dict_topic_relation_high_level_updated, reverse=reverse)
    save_to_pickle(subset_edges_high_level_updated, f'data/processed/subset_edges_high_level_m_{i}.pkl')
    reverse = not reverse



In [50]:
subset_edges_high_level_num_all_iters = []
for i in np.arange(0, 6, 1):
    subset_edges_high_level = load_from_pickle(f'data/processed/subset_edges_high_level_m_{i}.pkl')
    subset_edges_high_level_num_all_iters.append(len(subset_edges_high_level))

plt.figure(figsize=(4,3))
plt.plot(np.arange(0, 6, 1), subset_edges_high_level_num_all_iters, marker='o', linestyle='-', color='b')
plt.xlabel('Iteration')
plt.ylabel('Number of Subset Edges')
plt.title('Subset Edges Number Over Iterations')
plt.grid(True)



In [51]:
selected_iter = 2
subset_edges_high_level_selected = load_from_pickle(f'data/processed/subset_edges_high_level_m_{selected_iter}.pkl')

In [52]:
subset_edges_high_level_cycle_removed, removed_edges, unresolved_cycles, rounds = (
    fix_cycles_collect_unresolved(subset_edges_high_level_selected, verbose=True)
)


print(f"edges before cycle removal:",
        len(subset_edges_high_level_selected))
print(f"edges after cycle removal:",
        len(subset_edges_high_level_cycle_removed))
print(f"unresolved cycles:",
        unresolved_cycles)


In [53]:
subset_edges_high_level_direct = remove_transitive_edges(subset_edges_high_level_cycle_removed)

save_to_pickle(
        subset_edges_high_level_direct,
        "data/processed/subset_edges_high_level_direct_m.pkl")



In [54]:
original_to_shown_high_level = load_from_pickle('data/processed/original_to_shown_high_level_m.pkl')
topic_name_list_final = list(set([original_to_shown_high_level.get(node, node) for node in topic_name_list_updated]))
print(len(topic_name_list), len(topic_name_list_updated), len(topic_name_list_final))


subset_edges_low_level_direct_updated = [(original_to_shown_high_level.get(c1, c1), original_to_shown_high_level.get(c2, c2)) for (c1, c2) in subset_edges_low_level_direct]
print(len(subset_edges_low_level_direct), len(subset_edges_low_level_direct_updated))

all_edges = list(set(subset_edges_low_level_direct_updated + subset_edges_high_level_direct))

all_nodes_with_edges = set([c for (c1, c2) in all_edges for c in [c1, c2]])
all_nodes_without_edges = set(topic_name_list_final) - all_nodes_with_edges
print(len(all_nodes_with_edges), len(all_nodes_without_edges), len(topic_name_list_final))

sub_roots_all_edges, nonroot_nodes_all_edges = get_sub_hierarchies_root_nodes(all_edges)
print(len(sub_roots_all_edges), len(nonroot_nodes_all_edges))
for c in sub_roots_all_edges:
    all_edges.append((c, 'Methodological Innovation'))

for c in all_nodes_without_edges:
    all_edges.append((c, 'Methodological Innovation'))

print(len(all_edges), len(set([c for (c1, c2) in all_edges for c in [c1, c2]])))


all_edges = remove_transitive_edges(all_edges)
print(len(all_edges))


In [55]:
save_to_pickle(all_edges, 'data/processed/all_edges_m.pkl')

In [56]:
os.makedirs('data/results', exist_ok=True)
write_hierarchy_to_file(all_edges, 'data/results/hierarchy_m.txt')

In [57]:
name2ids = load_from_pickle('data/processed/name2ids_m.pkl')
shown_to_originals_high_level = load_from_pickle('data/processed/shown_to_originals_high_level_m.pkl')
shown_to_originals_low_level = load_from_pickle('data/processed/shown_to_originals_low_level_m.pkl')


dict_topic_keywords = defaultdict(set)
for topic in topic_name_list_final:
    if topic in shown_to_originals_high_level:
        t1s = shown_to_originals_high_level[topic]
    else:
        t1s = [topic]
    for t1 in t1s:
        if t1 in shown_to_originals_low_level:
            t2s = shown_to_originals_low_level[t1]
        else:
            t2s = [t1]
        for t2 in t2s:
            if t2 in name2ids:
                ids = name2ids[t2]
                for i in ids:
                    kws = set(dict_topic_keyword_proportion[i].keys())
                    dict_topic_keywords[topic].update(kws)
            else:
                print(topic, "--", t1, "--", t2)
save_to_pickle(dict_topic_keywords, 'data/processed/dict_topic_keywords_m.pkl')
print(len(dict_topic_keywords))